In [ ]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath("../../"))

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.path.join(os.getenv("ROOT_DIR"), ".data_xuetangx/XuetangX/raw")
OUTPUT_DIR = os.path.join(ROOT_DIR, "processed")

In [2]:
def create_full_grid(enroll_ids, max_day):
    """
    Crea una griglia perfetta (Studente x Giorno) per garantire che 
    non manchi nessun giorno nella sequenza temporale (necessario per l'LSTM).
    """
    days = np.arange(1, max_day + 1)
    multi_idx = pd.MultiIndex.from_product([enroll_ids, days], names=['enroll_id', 'day'])
    return pd.DataFrame(index=multi_idx).reset_index()


def process_xuetang_split(log_path, truth_path, course_info_path, all_actions=None):
    """
    Esplode i log di XuetangX trasformandoli nel formato Unitelma:
    1 riga = 1 studente per 1 giorno. Colonne = Conteggio Azioni.
    """
    print(f"[INFO] Loading file -> {os.path.basename(log_path)}...")
    
    df_log = pd.read_csv(log_path)
    df_truth = pd.read_csv(truth_path)
    df_course = pd.read_csv(course_info_path)
    
    df_log = df_log.merge(df_course[['course_id', 'start']], on='course_id', how='left')
    
    df_log['time'] = pd.to_datetime(df_log['time'])
    df_log['start'] = pd.to_datetime(df_log['start'])
    
    df_log['day'] = (df_log['time'] - df_log['start']).dt.days + 1
    
    df_log = df_log[df_log['day'] > 0]
    
    daily_actions = df_log.groupby(['enroll_id', 'day', 'action']).size().reset_index(name='count')
    
    pivot_df = daily_actions.pivot_table(
        index=['enroll_id', 'day'], 
        columns='action', 
        values='count', 
        fill_value=0
    ).reset_index()
    
    if all_actions is not None:
        for action in all_actions:
            if action not in pivot_df.columns:
                pivot_df[action] = 0
        pivot_df = pivot_df[['enroll_id', 'day'] + all_actions]
    else:
        all_actions = [col for col in pivot_df.columns if col not in ['enroll_id', 'day']]
    
    max_day = 30
    enroll_ids = df_truth['enroll_id'].unique()
    
    print(f"[INFO] Padding...")
    full_grid = create_full_grid(enroll_ids, max_day)
    
    df_final = full_grid.merge(pivot_df, on=['enroll_id', 'day'], how='left').fillna(0)
    
    for col in all_actions:
        df_final[col] = df_final[col].astype(int)
        
    df_final = df_final.merge(df_truth, on='enroll_id', how='inner')
    
    df_final.rename(columns={'truth': 'dropout', 'enroll_id': 'global_id'}, inplace=True)
    
    df_final = df_final.sort_values(['global_id', 'day']).reset_index(drop=True)
    
    print(f"[INFO] Complete, Shape: {df_final.shape}")
    return df_final, all_actions


def build_and_save_xuetang_flat(root_dir, output_dir):
    """
    Funzione principale da lanciare nel notebook.
    Coordina train e test e salva i CSV finali pronti per l'LSTM.
    """
    print("[INFO] Inizio conversione dataset XuetangX formato Unitelma (Esploso)")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Percorsi dei file originali
    train_log = os.path.join(root_dir, "prediction_log", "train_log.csv")
    train_truth = os.path.join(root_dir, "prediction_log", "train_truth.csv")
    test_log = os.path.join(root_dir, "prediction_log", "test_log.csv")
    test_truth = os.path.join(root_dir, "prediction_log", "test_truth.csv")
    course_info = os.path.join(root_dir, "course_info.csv")
    
    # 1. Processiamo il Train
    print("\n[INFO] Processing Train Set...")
    df_train, found_actions = process_xuetang_split(
        train_log, train_truth, course_info
    )
    
    # 2. Processiamo il Test (passando le azioni trovate nel train per allineare le colonne)
    print("\n[INFO] Processing Test Set...")
    df_test, _ = process_xuetang_split(
        test_log, test_truth, course_info, all_actions=found_actions
    )
    
    # 3. Salvataggio
    train_out = os.path.join(output_dir, "xuetang_flat_train.csv")
    test_out = os.path.join(output_dir, "xuetang_flat_test.csv")
    
    df_train.to_csv(train_out, index=False)
    df_test.to_csv(test_out, index=False)
    
    print(f"\n[SUCCESS] Dataset esploso salvato in: {output_dir}")
    print(f"Features estratte ({len(found_actions)}): {found_actions}")
    
    return df_train, df_test

In [3]:
df_train_flat, df_test_flat = build_and_save_xuetang_flat(ROOT_DIR, OUTPUT_DIR)

[INFO] Inizio conversione dataset XuetangX formato Unitelma (Esploso)

[INFO] Processing Train Set...
[INFO] Loading file -> train_log.csv...
[INFO] Padding...
[INFO] Complete, Shape: (4738290, 25)

[INFO] Processing Test Set...
[INFO] Loading file -> test_log.csv...
[INFO] Padding...
[INFO] Complete, Shape: (2030970, 25)

[SUCCESS] Dataset esploso salvato in: /Volumes/T7/documents/github/dropout-prediction/.data_xuetangx/XuetangX/raw/processed
Features estratte (22): ['click_about', 'click_courseware', 'click_forum', 'click_info', 'click_progress', 'close_courseware', 'close_forum', 'create_comment', 'create_thread', 'delete_comment', 'delete_thread', 'load_video', 'pause_video', 'play_video', 'problem_check', 'problem_check_correct', 'problem_check_incorrect', 'problem_get', 'problem_save', 'reset_problem', 'seek_video', 'stop_video']
